<a href="https://colab.research.google.com/github/alnmuniz/pln-projeto-final/blob/main/projeto_final_pln_grupo_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto 3: Chatbot RAG para Perguntas e Respostas sobre um Livro

**Disciplina:** Processamento de Linguagem Natural e IA Generativa

**Professor:** Márcio Freire

**Data**: Março de 2026

**Grupo**:

* André Luís Nunes Muniz
* Victor Fernandes Baião Raton
* Vinicius dos Anjos Pinto

Este notebook implementa uma arquitetura completa de Retrieval-Augmented Generation (RAG) para responder a perguntas baseadas no conteúdo de livros de domínio público.

**Bloco de Configuração:**
Para flexibilizar o projeto, implementamos um gerenciador de configurações em JSON logo no início do notebook. Isso permite alternar dinamicamente todo o contexto do chatbot (PDF, banco vetorial, prompts e interface) entre diferentes livros (como 'O Capital' e '1984'), sem a necessidade de reescrever ou duplicar o código nas etapas seguintes.

In [1]:
# Bloco 0: Configurações Gerais do Projeto
import json

# =======================================================
# ESCOLHA O LIVRO AQUI: Mude para 'marx' ou 'orwell'
# =======================================================
LIVRO_ATUAL = 'orwell'

if LIVRO_ATUAL == 'marx':
    CONFIG = {
        "drive_id": "1vjb0YTkt8xuIsBKazk9tnW3v3h4F5Cl0",
        "nome_pdf": "o_capital.pdf",
        "nome_colecao": "o_capital_marx",
        "caminho_qdrant": "./qdrant_marx_db",
        "pergunta_teste": "O que é a mais-valia e como o capitalista lucra com ela?",
        "prompt_sistema": "Você é um assistente acadêmico especialista na obra 'O Capital' de Karl Marx. Responda à pergunta do usuário utilizando ÚNICA E EXCLUSIVAMENTE as informações fornecidas no contexto abaixo. Se a resposta não estiver no contexto, diga que não sabe com base nos trechos fornecidos.",
        "titulo_interface": "🤖 Assistente de 'O Capital' (Karl Marx)",
        "icone_interface": "📕"
    }
elif LIVRO_ATUAL == 'orwell':
    CONFIG = {
        "drive_id": "1oQ6DNS81A2zvXjZmWXquK-Y6P4nXE5pO",
        "nome_pdf": "1984_orwell.pdf",
        "nome_colecao": "1984_orwell",
        "caminho_qdrant": "./qdrant_1984_db",
        "pergunta_teste": "O que é o Duplipensar (Doublethink) e qual o seu objetivo para o Partido?",
        "prompt_sistema": "Você é um assistente acadêmico especialista na obra '1984' de George Orwell. Responda à pergunta do usuário utilizando ÚNICA E EXCLUSIVAMENTE as informações fornecidas no contexto abaixo. Se a resposta não estiver no contexto, diga que não sabe com base nos trechos fornecidos.",
        "titulo_interface": "🤖 Assistente de '1984' (George Orwell)",
        "icone_interface": "👁️"
    }
else:
    raise ValueError("Livro não reconhecido. Escolha 'marx' ou 'orwell'.")

# Salvando as configurações em disco para os outros blocos e o Streamlit acessarem
with open("config_projeto.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=4)

print(f"✅ Configurações carregadas e prontas para o projeto: {CONFIG['titulo_interface']}")

✅ Configurações carregadas e prontas para o projeto: 🤖 Assistente de '1984' (George Orwell)


## Etapa 1: Ingestão e Pré-processamento (Chunking)

O processamento de dados não estruturados, como PDFs, apresenta desafios técnicos devido à formatação irregular. Para contornar isso, se torna necessário de *chunking* (divisão de documentos grandes em pedaços menores) como um passo técnico essencial.

Nesta etapa, o texto extraído é limpo e dividido utilizando o `RecursiveCharacterTextSplitter`, garantindo que pedaços da informação mantenham a coesão semântica através de uma sobreposição (overlap) de caracteres para não perder o contexto entre as frases.

***

In [2]:
# Bloco 1: Instalação de pacotes e processamento do texto
!pip install -q pypdf langchain-text-splitters

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
import gdown
import json

with open("config_projeto.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

print(f"Baixando o livro {CONFIG['nome_pdf']} do Google Drive...")
url = f"https://drive.google.com/uc?id={CONFIG['drive_id']}"
gdown.download(url, CONFIG['nome_pdf'], quiet=False)

caminho_pdf = CONFIG['nome_pdf']


def extrair_e_limpar_texto(caminho_arquivo):
    print("Lendo o PDF e extraindo o texto...")
    leitor = PdfReader(caminho_arquivo)
    texto_completo = ""

    for pagina in leitor.pages:
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            texto_completo += texto_pagina

    # Limpeza básica: remove quebras de linha no meio de frases e espaços duplos
    texto_limpo = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto_completo)
    texto_limpo = re.sub(r'\s+', ' ', texto_limpo).strip()
    print(f"Sucesso! Texto extraído com {len(texto_limpo)} caracteres.")
    return texto_limpo

def criar_chunks(texto):
    print("Dividindo o texto em chunks...")
    # O divisor tenta quebrar por parágrafos duplos, depois simples, depois pontos...
    divisor = RecursiveCharacterTextSplitter(
        chunk_size=600,   # Tamanho de cada pedaço de texto
        chunk_overlap=50, # Sobreposição para não perder o contexto entre um chunk e outro
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = divisor.split_text(texto)
    print(f"Total de {len(chunks)} chunks gerados.")
    return chunks

# Execução do pipeline da Etapa 1
try:
    texto_livro = extrair_e_limpar_texto(caminho_pdf)
    chunks_livro = criar_chunks(texto_livro)

    # Exibindo uma amostra para verificação
    print("\n--- Amostra do 10º chunk gerado ---")
    print(chunks_livro[9])
except FileNotFoundError:
    print("ERRO: O arquivo 'o_capital.pdf' não foi encontrado. Verifique se o upload terminou no painel lateral do Colab.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.8 MB/s eta 0:00:00
Baixando o livro 1984_orwell.pdf do Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1oQ6DNS81A2zvXjZmWXquK-Y6P4nXE5pO
To: /content/1984_orwell.pdf
100%|██████████| 2.54M/2.54M [00:00<00:00, 156MB/s]


Lendo o PDF e extraindo o texto...
Sucesso! Texto extraído com 709333 caracteres.
Dividindo o texto em chunks...
Total de 1386 chunks gerados.

--- Amostra do 10º chunk gerado ---
. A um quilômetro de distância, o M inistério da Verdade, onde ele trabalhava, erguia-se vasto e branco por sobre a paisagem encardida. Aquela, pensou com uma espécie de contrariedade difusa, aquela era Londres, principal cidade da Faixa Aérea Um, terceira mais populosa das províncias da Oceânia. Tentou localizar alguma lembrança de infância que lhe dissesse se Londres sempre fora assim


## Etapa 2: Vetorização e Indexação no Banco Vetorial

Para atender aos requisitos de não utilizar os mesmos componentes do projeto 4, selecionamos as alternativas a seguir:

* **Modelo de Embedding (`intfloat/multilingual-e5-base`):** Embora o topo absoluto do [ranking MTEB](https://huggingface.co/spaces/mteb/leaderboard) seja atualmente dominado por modelos gigantes (com 7 a 12 bilhões de parâmetros), o E5-base destaca-se como um dos modelos leves de código aberto mais eficientes. Ele foi estrategicamente escolhido por superar outras opções leves semelhantes e também superar o MiniLM visto em sala na semântica em português, mantendo um consumo de memória otimizado (aproximadamente 278 milhões de parâmetros). Isso é fundamental na nossa arquitetura para garantir que o modelo de embedding possa rodar simultaneamente com o LLM gerador dentro do limite de 16 GB de VRAM da GPU gratuita do Google Colab (T4), evitando o esgotamento da memória (CUDA out of memory).
* **Banco Vetorial (`Qdrant`):** Como FAISS, Chroma e Pinecone estão vetados, o Qdrant foi escolhido. Ele é rápido, focado em busca vetorial e suporta persistência local em disco diretamente no Colab, sem precisar de servidores em nuvem.

**Otimização de Performance:** Em vez de processar o texto sequencialmente, implementamos uma função que utiliza processamento em lote (*batching*) na GPU. Isso reduz o tempo de vetorização, enviando dezenas de sentenças simultaneamente para a placa de vídeo.

***

In [3]:
# Bloco 2: Instalação, Vetorização (E5) e Indexação (Qdrant)
!pip install -q qdrant-client sentence-transformers

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import uuid
import json

# 1. SEGURANÇA: Fechar conexões antigas do notebook para evitar erro de 'AlreadyLocked'
if 'qdrant' in locals():
    try:
        qdrant.close()
        print("Conexão antiga do Qdrant fechada.")
    except:
        pass

with open("config_projeto.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

# 2. Inicializar o cliente do Qdrant
print("Conectando ao banco Qdrant local...")
qdrant = QdrantClient(path=CONFIG['caminho_qdrant'])
NOME_COLECAO = CONFIG['nome_colecao']

# 3. Inicializar o modelo de embeddings E5
print("Carregando o modelo de embeddings...")
modelo_embedding = SentenceTransformer("intfloat/multilingual-e5-base")
DIMENSAO_VETOR = modelo_embedding.get_sentence_embedding_dimension()

# 4. Criar a coleção no Qdrant (apagando a antiga se já existir)
if qdrant.collection_exists(collection_name=NOME_COLECAO):
    print(f"A coleção '{NOME_COLECAO}' já existe. Apagando a versão antiga para recriar...")
    qdrant.delete_collection(collection_name=NOME_COLECAO)

qdrant.create_collection(
    collection_name=NOME_COLECAO,
    vectors_config=VectorParams(size=DIMENSAO_VETOR, distance=Distance.COSINE),
)
print(f"Coleção '{NOME_COLECAO}' pronta para receber os vetores!")

# 5. Função para vetorizar e indexar os chunks (Processamento em Lote/Batch)
def indexar_chunks(chunks, cliente_qdrant, modelo, nome_colecao, tamanho_lote_gpu=64):
    print(f"\nIniciando a vetorização em lotes de {len(chunks)} chunks...")

    # Prepara todos os textos com o prefixo exigido de uma só vez
    textos_para_vetorizar = [f"passage: {chunk}" for chunk in chunks]

    # Manda a lista inteira para a GPU processar em paralelo
    print(f"Gerando matrizes de embeddings na GPU (Lotes de {tamanho_lote_gpu})...")
    vetores = modelo.encode(
        textos_para_vetorizar,
        batch_size=tamanho_lote_gpu,
        show_progress_bar=True
    ).tolist()

    # Preparando e enviando para o Qdrant
    print("Enviando vetores para o banco Qdrant...")
    pontos = []
    for i, (chunk, vetor) in enumerate(zip(chunks, vetores)):
        ponto = PointStruct(
            id=str(uuid.uuid4()),
            vector=vetor,
            payload={"texto": chunk, "id_chunk": i}
        )
        pontos.append(ponto)

        # Upsert em lotes de 200 no banco de dados
        if len(pontos) >= 200 or i == len(chunks) - 1:
            cliente_qdrant.upsert(
                collection_name=nome_colecao,
                points=pontos
            )
            pontos = [] # Limpa a lista para o próximo lote

    print("Indexação concluída com sucesso!")

# 6. Executar a vetorização
if 'chunks_livro' in locals() and len(chunks_livro) > 0:
    indexar_chunks(chunks_livro, qdrant, modelo_embedding, NOME_COLECAO)

    info = qdrant.get_collection(NOME_COLECAO)
    print(f"\nStatus final do Qdrant: {info.points_count} vetores armazenados perfeitamente.")
else:
    print("ERRO: A variável 'chunks_livro' não foi encontrada. Rode o Bloco 1 primeiro.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 8.2 MB/s eta 0:00:00
Conectando ao banco Qdrant local...
Carregando o modelo de embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Coleção '1984_orwell' pronta para receber os vetores!

Iniciando a vetorização em lotes de 1386 chunks...
Gerando matrizes de embeddings na GPU (Lotes de 64)...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Enviando vetores para o banco Qdrant...
Indexação concluída com sucesso!

Status final do Qdrant: 1386 vetores armazenados perfeitamente.


## Etapa 3: Pipeline de Recuperação (Retriever)

Na prática base fornecida na disciplina, a busca foi feita criando vetores e comparando distâncias usando a biblioteca FAISS.

Nesta arquitetura atualizada, utilizamos a nova API do Qdrant para realizar matematicamente o mesmo processo de busca por similaridade (distância de cosseno) para encontrar as partes mais relevantes do documento. Para maximizar a precisão do modelo E5 escolhido, formatamos as perguntas do usuário com o prefixo obrigatório `query:`.

***

In [4]:
# Bloco 3: Pipeline de Recuperação (Buscador no Qdrant)

import json

# Lendo a pergunta de teste das configurações
with open("config_projeto.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

pergunta_teste = CONFIG['pergunta_teste']

def buscar_contexto(pergunta, cliente_qdrant, modelo, nome_colecao, top_k=3):
    print(f"Buscando as {top_k} passagens mais relevantes no livro...")

    # 1. Aplicar o prefixo obrigatório do modelo E5 para perguntas
    pergunta_formatada = f"query: {pergunta}"

    # 2. Transformar a pergunta em vetor (embedding)
    vetor_pergunta = modelo.encode(pergunta_formatada).tolist()

    # 3. Fazer a busca de similaridade (Cosseno) no Qdrant (NOVA API)
    resposta = cliente_qdrant.query_points(
        collection_name=nome_colecao,
        query=vetor_pergunta,
        limit=top_k
    )

    # A nova API retorna um objeto, a lista de resultados fica no atributo .points
    resultados = resposta.points

    # 4. Extrair e retornar apenas os textos recuperados
    contextos_recuperados = []
    print("\nResultados encontrados:")
    for i, res in enumerate(resultados):
        texto = res.payload['texto']
        score = res.score
        contextos_recuperados.append(texto)

        # Imprime um resumo do que encontrou para conferência
        print(f"\n--- Top {i+1} (Similaridade: {score:.4f}) ---")
        print(f"{texto[:250]}...") # Mostra os primeiros 250 caracteres

    return contextos_recuperados

# ==========================================
# TESTE DO BUSCADOR
# ==========================================
print(f"Pergunta de teste: '{pergunta_teste}'\n")

# Vamos buscar os 4 chunks mais relevantes
contextos = buscar_contexto(pergunta_teste, qdrant, modelo_embedding, NOME_COLECAO, top_k=4)

Pergunta de teste: 'O que é o Duplipensar (Doublethink) e qual o seu objetivo para o Partido?'

Buscando as 4 passagens mais relevantes no livro...

Resultados encontrados:

--- Top 1 (Similaridade: 0.8961) ---
. Tal como descrito em Teoria e prática do coletivism o oligárquico, de Emmanuel Goldstein, um texto perigosamente subversivo, proscrito na Oceânia e conhecido somente como o livro, o duplipensamento é uma forma de disciplina mental cujo objetivo, de...

--- Top 2 (Similaridade: 0.8817) ---
. Duplipensam ento signi ca a capacidade de abrigar simultaneamente na cabeça duas crenças contraditórias e acreditar em ambas. O intelectual do Partido sabe em que direção suas memórias precisam ser alteradas; em consequência, sabe que está manipula...

--- Top 3 (Similaridade: 0.8802) ---
. O duplipensam ento situa-se no âmago do Socing, visto que o ato essencial do Partido consiste em usar o engodo consciente sem perder a  rmeza de propósito que corresponde à total honestidade. Dizer menti

## Etapa 4: Geração Aumentada (LLM)

O projeto aplica a teoria de que o RAG resolve o problema do conhecimento estático dos modelos de linguagem. O LLM não precisa ter memorizado a obra, pois o contexto exato é fornecido na hora da requisição.

* **Modelo Escolhido (`Qwen/Qwen2.5-1.5B-Instruct` - [ver mais](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)):** O modelo `google/flan-t5-base` utilizado em sala e o `bloom-560m` foi escolhido pela outra equipe, impedindo a sua seleção para este trabalho. A arquitetura Qwen 2.5 moderna apresenta excelente suporte ao português e capacidade superior de seguir instruções sistêmicas, essencial para focar estritamente no texto e evitar alucinações.
A função que antes preparava o contexto para o FLAN-T5 do roteiro feito em sala foi adaptado para inicializar a arquitetura do Qwen (via `AutoModelForCausalLM`).

***

In [5]:
# Bloco 4: Geração Aumentada (RAG) com o Qwen 2.5
!pip install -q accelerate transformers torch

import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

with open("config_projeto.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

# Definindo o modelo escolhido
NOME_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Carregando o modelo {NOME_LLM} (isso vai baixar cerca de 3GB)...")

# Carregando o Tokenizador e o Modelo na GPU (float16 economiza muita memória)
tokenizer_llm = AutoTokenizer.from_pretrained(NOME_LLM)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    NOME_LLM,
    torch_dtype=torch.float16,
    device_map="auto" # Manda automaticamente para a GPU se disponível
)
print("LLM carregado com sucesso na GPU!")

def gerar_resposta_qwen(pergunta, contextos, modelo, tokenizer):
    # 1. Juntar os blocos de texto recuperados do Qdrant em um único texto contínuo
    contexto_completo = "\n\n---\n\n".join(contextos)

    # 2. Montar a instrução do sistema e do usuário (Engenharia de Prompt)
    mensagens = [
        {"role": "system", "content": CONFIG['prompt_sistema']},
        {"role": "user", "content": f"CONTEXTO RECUPERADO:\n{contexto_completo}\n\nPERGUNTA: {pergunta}"}
    ]

    # 3. Preparar o prompt no formato de chat que o Qwen entende
    texto_prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    # 4. Tokenizar e enviar os dados para a GPU
    inputs = tokenizer([texto_prompt], return_tensors="pt").to(modelo.device)

    # 5. Gerar a resposta
    print("\nO Qwen está analisando o texto e formulando a resposta...")
    with torch.no_grad():
        outputs = modelo.generate(
            **inputs,
            max_new_tokens=512, # Limite de tamanho da resposta
            temperature=0.3,    # Temperatura baixa (0.3) reduz a criatividade e foca nos fatos do texto (evita alucinações)
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 6. Decodificar apenas a parte gerada (removendo o prompt original que enviamos)
    tamanho_input = inputs.input_ids.shape[1]
    resposta_gerada = tokenizer.decode(outputs[0][tamanho_input:], skip_special_tokens=True)

    return resposta_gerada

# ==========================================
# TESTE DO RAG COMPLETO
# ==========================================
# Usaremos a variável 'contextos' e 'pergunta_teste' que criamos no Bloco 3
if 'contextos' in locals() and len(contextos) > 0:
    resposta_final = gerar_resposta_qwen(pergunta_teste, contextos, modelo_llm, tokenizer_llm)
    print("\n" + "="*60)
    print("🤖 RESPOSTA DO RAG:")
    print("="*60)
    print(resposta_final)
else:
    print("Aguardando o Bloco 3 para rodar o teste completo.")

Carregando o modelo Qwen/Qwen2.5-1.5B-Instruct (isso vai baixar cerca de 3GB)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM carregado com sucesso na GPU!

O Qwen está analisando o texto e formulando a resposta...

🤖 RESPOSTA DO RAG:
O Duplipensar (Doublethink) é uma técnica utilizada pelo Partido em 1984 para manter seus membros coerentes em sua ideologia e lógica. Seu objetivo principal é permitir que os membros do Partido mantenham múltiplas crenças conflitantes dentro de si, mantendo-os dispostos a crer em ambos enquanto continuam a seguir as regras impostas pela organização.

Além disso, o duplipensar também serve para controlar a realidade, permitindo que os membros do Partido mantenham certas informações ocultas ou inofensivas sob controle enquanto outras são completamente ignoradas ou revogadas conforme necessário. Isso permite que o Partido continue funcionando sem que seus membros percebam a natureza absolutamente falsa de suas crenças e comportamentos.


*texto em itálico*
## Etapa 5: Interface Gráfica e Exposição com Ngrok

A especificação permite utilizar a mesma interface com a biblioteca Gradio ou implementar uma nova com a biblioteca Streamlit. Optamos pelo Streamlit pela sua capacidade superior de gerenciar o estado da sessão (histórico de chat) de forma nativa.

**Engenharia de Confiabilidade (Tratamento de Exceções):**
Para conferir nível de produção à aplicação acadêmica, encapsulamos a lógica de geração com tratamento de erros (`try/except`). Caso a pergunta do usuário provoque um estouro de memória na GPU do Colab (`CUDA out of memory`), o aplicativo intercepta a falha, limpa o cache da placa de vídeo e exibe um alerta amigável na tela em vez de quebrar o sistema.

A aplicação web é exposta publicamente através de um túnel seguro usando o pacote `pyngrok`, com a chave de autenticação protegida pelos recursos de "Secrets" do Google Colab.

***

In [6]:
# Bloco 5.1: Instalações necessárias
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 47.2 MB/s eta 0:00:00


In [7]:
# Fechar a instância do Qdrant no notebook para libertar o acesso ao Streamlit
if 'qdrant' in locals():
    qdrant.close()
    print("Conexão do notebook com o Qdrant fechada com sucesso! O Streamlit já pode aceder à base de dados.")

Conexão do notebook com o Qdrant fechada com sucesso! O Streamlit já pode aceder à base de dados.


In [8]:
%%writefile app.py
import streamlit as st
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

with open("config_projeto.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

# 1. Configuração da Página
st.set_page_config(page_title="Chatbot RAG", page_icon=CONFIG['icone_interface'])
st.title(CONFIG['titulo_interface'])
st.write("Faça perguntas sobre a obra com base no texto indexado.")

# 2. Carregamento dos Modelos (com cache para não recarregar a cada interação)
@st.cache_resource
def carregar_recursos():
    # Inicializa o Qdrant apontando para o diretório salvo em disco
    qdrant = QdrantClient(path=CONFIG['caminho_qdrant'])

    # Carrega Embeddings
    modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")

    # Carrega LLM
    nome_llm = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(nome_llm)
    modelo_llm = AutoModelForCausalLM.from_pretrained(
        nome_llm, torch_dtype=torch.float16, device_map="auto"
    )

    return qdrant, modelo_emb, modelo_llm, tokenizer

qdrant, modelo_emb, modelo_llm, tokenizer = carregar_recursos()
NOME_COLECAO = CONFIG['nome_colecao']

# 3. Histórico do Chat
if "mensagens" not in st.session_state:
    st.session_state.mensagens = []

for msg in st.session_state.mensagens:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 4. Input do Usuário
pergunta = st.chat_input("Pergunte algo sobre a obra...")

if pergunta:
    # Exibe pergunta do usuário
    with st.chat_message("user"):
        st.markdown(pergunta)
    st.session_state.mensagens.append({"role": "user", "content": pergunta})

    with st.chat_message("assistant"):
        placeholder = st.empty()
        placeholder.markdown("Buscando passagens relevantes e formulando resposta...")

        # BLINDAGEM DO CÓDIGO COM TRY/EXCEPT
        try:
            # Recuperação (Retriever)
            pergunta_formatada = f"query: {pergunta}"
            vetor_pergunta = modelo_emb.encode(pergunta_formatada).tolist()

            resposta_busca = qdrant.query_points(
                collection_name=NOME_COLECAO, query=vetor_pergunta, limit=4
            )

            contextos = [res.payload['texto'] for res in resposta_busca.points]
            contexto_completo = "\n\n---\n\n".join(contextos)

            # Geração (LLM)
            mensagens_llm = [
                {"role": "system", "content": CONFIG['prompt_sistema']},
                {"role": "user", "content": f"CONTEXTO:\n{contexto_completo}\n\nPERGUNTA: {pergunta}"}
            ]

            texto_prompt = tokenizer.apply_chat_template(mensagens_llm, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer([texto_prompt], return_tensors="pt").to(modelo_llm.device)

            with torch.no_grad():
                outputs = modelo_llm.generate(**inputs, max_new_tokens=512, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)

            tamanho_input = inputs.input_ids.shape[1]
            resposta = tokenizer.decode(outputs[0][tamanho_input:], skip_special_tokens=True)

            # Mostra resposta e fontes
            resposta_final = f"{resposta}\n\n**Trechos utilizados como base:**\n"
            for i, txt in enumerate(contextos):
                resposta_final += f"- *{txt[:150]}...*\n"

            placeholder.markdown(resposta_final)
            st.session_state.mensagens.append({"role": "assistant", "content": resposta_final})

        except Exception as e:
            # Limpa o "Buscando..." e exibe uma mensagem amigável no lugar
            erro_msg = "⚠️ **Desculpe, o sistema encontrou um problema ao processar sua pergunta.** "

            # Tratamento específico para falta de memória na placa de vídeo
            if "CUDA out of memory" in str(e):
                erro_msg += "A pergunta ou o contexto excederam o limite de memória da GPU. Por favor, faça uma pergunta mais curta."
                # Limpa o cache da GPU para tentar recuperar a estabilidade
                torch.cuda.empty_cache()
            else:
                erro_msg += "Tente novamente em instantes."

            placeholder.error(erro_msg)
            # Mostra o log técnico no terminal do Colab, mas esconde do usuário final
            print(f"Erro na geração RAG: {e}")

Writing app.py


In [ ]:
# Bloco 5.3: Executar a Aplicação com Ngrok
!pip install -q pyngrok

from pyngrok import ngrok
from google.colab import userdata
import time

# 1. Autenticação buscando o token com segurança no Colab Secrets
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
except userdata.SecretNotFoundError:
    print("ERRO: O segredo 'NGROK_TOKEN' não foi encontrado.")
    print("Por favor, crie-o no menu lateral esquerdo (ícone de chave) e ative o 'Notebook access'.")
    raise

# 2. Fecha túneis antigos para evitar conflitos
ngrok.kill()

# 3. Cria um túnel seguro para a porta 8501 (porta padrão do Streamlit)
url_ngrok = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🔗 CLIQUE NESTE LINK PARA ACESSAR O CHATBOT:\n{url_ngrok}")
print("=" * 60)

# 4. Inicia o Streamlit
!streamlit run app.py

🔗 CLIQUE NESTE LINK PARA ACESSAR O CHATBOT:
https://nonexultantly-escapable-charlott.ngrok-free.dev



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.74.212.28:8501

Loading weights: 100% 199/199 [00:00<00:00, 656.17it/s, Materializing param=pooler.dense.weight]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:12<00:00, 27.10it/s, Materializing param=model.norm.weight]
